In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/achiver7/project-phase-1/data/sc_val.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/acts_val.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/acts_train.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/tribunals_val.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/test.json
/kaggle/input/datasets/achiver7/project-phase-1/data/sc_train.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/tribunals_train.jsonl
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/sc_train.bin
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/acts_train.bin
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/acts_val.bin
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/tribunals_val_f.bin
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/tokenizer.json
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/tribunals_train_f.bin
/kaggle/input/datasets/achive

In [2]:
# ============================================================
# PHASE 1 — REAL TRAINING ONLY
# NO SMOKE TEST
# NO BENCHMARK
# STARTS FRESH UNLESS RUN/latest.pt EXISTS
# ============================================================

import os
import time
import math
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# 0. YOUR MODEL
# ============================================================

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size):
        super().__init__()

        assert d_model % n_heads == 0

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        mask = torch.tril(
            torch.ones(block_size, block_size)
        ).view(1, 1, block_size, block_size)

        self.register_buffer("mask", mask)

    def forward(self, x):
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B, T, self.n_heads, self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B, T, self.n_heads, self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B, T, self.n_heads, self.head_dim
        ).transpose(1, 2)

        att = (
            q @ k.transpose(-2, -1)
        ) / math.sqrt(self.head_dim)

        att = att.masked_fill(
            self.mask[:, :, :T, :T] == 0,
            float("-inf")
        )

        att = F.softmax(att, dim=-1)

        y = att @ v

        y = (
            y.transpose(1, 2)
             .contiguous()
             .view(B, T, C)
        )

        return self.out_proj(y)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()

        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(
            F.gelu(self.fc1(x))
        )


class TransformerBlock(nn.Module):
    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        block_size
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(d_model)

        self.attn = CausalSelfAttention(
            d_model,
            n_heads,
            block_size
        )

        self.norm2 = nn.LayerNorm(d_model)

        self.mlp = FeedForward(
            d_model,
            d_ff
        )

    def forward(self, x):
        x = x + self.attn(
            self.norm1(x)
        )

        x = x + self.mlp(
            self.norm2(x)
        )

        return x


class LegalDecoderLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        block_size,
        d_model,
        n_layers,
        n_heads,
        d_ff
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.block_size = block_size
        self.d_model = d_model

        self.token_emb = nn.Embedding(
            vocab_size,
            d_model
        )

        self.pos_emb = nn.Embedding(
            block_size,
            d_model
        )

        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model,
                n_heads,
                d_ff,
                block_size
            )
            for _ in range(n_layers)
        ])

        self.final_norm = nn.LayerNorm(
            d_model
        )

        self.lm_head = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        self.lm_head.weight = (
            self.token_emb.weight
        )

    def forward(self, x):
        B, T = x.shape

        assert T <= self.block_size

        tok = self.token_emb(x)

        pos = self.pos_emb(
            torch.arange(
                T,
                device=x.device
            )
        )

        h = tok + pos

        for block in self.blocks:
            h = block(h)

        h = self.final_norm(h)

        return self.lm_head(h)


def build_model(vocab_size, block_size):
    d_model = 768
    n_layers = 8
    n_heads = 8
    d_ff = 3072

    return LegalDecoderLM(
        vocab_size=vocab_size,
        block_size=block_size,
        d_model=d_model,
        n_layers=n_layers,
        n_heads=n_heads,
        d_ff=d_ff
    )
# ============================================================
# 1. PATHS / CONFIG
# ============================================================

TOKD = "/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens"

# Use a unique run name for this experiment
RUN = "/kaggle/working/run_arch1_real"

V = 40000
TIME_LIMIT_H = 8.25
dev = "cuda"

CFG = dict(
    BLOCK=1024,
    MICRO_BATCH=1,
    GRAD_ACCUM=16,

    LR=3e-4,
    MIN_LR_FRAC=0.1,
    WD=0.1,
    WARMUP=500,

    STEPS=10000,
    CLIP=1.0,
    EVAL_EVERY=500,
    SEED=42,

    MIX=dict(
        acts=0.273,
        sc=0.469,
        tri=0.258
    )
)

# ============================================================
# 2. BASIC CHECKS
# ============================================================

assert torch.cuda.is_available(), "CUDA GPU required"
assert os.path.exists(TOKD), f"Missing: {TOKD}"
assert os.path.exists(f"{TOKD}/tokenizer.json")
assert callable(build_model), "build_model() not found"

assert abs(sum(CFG["MIX"].values()) - 1.0) < 1e-6

BLOCK = CFG["BLOCK"]
MB = CFG["MICRO_BATCH"]
GA = CFG["GRAD_ACCUM"]

np.random.seed(CFG["SEED"])
torch.manual_seed(CFG["SEED"])
torch.cuda.manual_seed_all(CFG["SEED"])

os.makedirs(RUN, exist_ok=True)

print("=" * 70)
print("PHASE 1 — REAL TRAINING")
print("=" * 70)
print("GPU:", torch.cuda.get_device_name(0))
print(
    "VRAM:",
    round(
        torch.cuda.get_device_properties(0).total_memory / 1e9,
        2
    ),
    "GB"
)
print("BLOCK:", BLOCK)
print("MICRO_BATCH:", MB)
print("GRAD_ACCUM:", GA)
print(
    "TOKENS / OPTIMIZER STEP:",
    MB * GA * BLOCK
)
print("TOTAL STEPS:", CFG["STEPS"])
print("TIME LIMIT:", TIME_LIMIT_H, "hours")
print()

# ============================================================
# 3. TOKEN DATA
# ============================================================

def mm(name):
    path = f"{TOKD}/{name}.bin"
    assert os.path.exists(path), f"Missing token file: {path}"
    return np.memmap(
        path,
        dtype=np.uint16,
        mode="r"
    )

data = {
    "acts": mm("acts_train"),
    "sc": mm("sc_train"),
    "tri": mm("tribunals_train_f"),
}

valbins = {
    "acts_val": mm("acts_val"),
    "sc_val": mm("sc_val"),
    "tribunals_val_f": mm("tribunals_val_f"),
}

keys = list(CFG["MIX"].keys())
probs = list(CFG["MIX"].values())

print("TRAIN TOKENS")
for name, d in data.items():
    print(f"{name:5s}: {len(d):,}")

print()

# ============================================================
# 4. BATCHING
# ============================================================

def get_batch():
    xs = []
    ys = []

    for _ in range(MB):

        source = keys[
            np.random.choice(
                len(keys),
                p=probs
            )
        ]

        d = data[source]

        i = np.random.randint(
            0,
            len(d) - BLOCK - 1
        )

        chunk = d[
            i:i + BLOCK + 1
        ].astype(np.int64)

        xs.append(chunk[:-1])
        ys.append(chunk[1:])

    x = torch.from_numpy(
        np.stack(xs)
    ).to(dev)

    y = torch.from_numpy(
        np.stack(ys)
    ).to(dev)

    return x, y

# ============================================================
# 5. FRESH MODEL
# ============================================================

print("Building model...")

model = build_model(V, BLOCK).to(dev)

n_params = sum(
    p.numel()
    for p in model.parameters()
)

print(
    f"PARAMETERS: {n_params:,}"
)
print(
    f"PARAMETERS: {n_params / 1e6:.2f}M"
)

assert 50e6 <= n_params <= 300e6, (
    f"Invalid parameter count: {n_params/1e6:.2f}M"
)

# ============================================================
# 6. OPTIMIZER + AMP
# ============================================================

opt = torch.optim.AdamW(
    model.parameters(),
    lr=CFG["LR"],
    weight_decay=CFG["WD"]
)

scaler = torch.amp.GradScaler("cuda")

# ============================================================
# 7. LEARNING RATE
# ============================================================

def lr_at(step):

    if step < CFG["WARMUP"]:
        return (
            CFG["LR"]
            * (step + 1)
            / CFG["WARMUP"]
        )

    p = (
        step - CFG["WARMUP"]
    ) / max(
        1,
        CFG["STEPS"] - CFG["WARMUP"]
    )

    p = min(
        max(p, 0.0),
        1.0
    )

    return CFG["LR"] * (
        CFG["MIN_LR_FRAC"]
        + (
            1 - CFG["MIN_LR_FRAC"]
        )
        * 0.5
        * (
            1 + math.cos(
                math.pi * p
            )
        )
    )

# ============================================================
# 8. FIXED VALIDATION WINDOWS
# ============================================================

rs = np.random.RandomState(123)

val_windows = {}

for name, d in valbins.items():

    starts = rs.randint(
        0,
        len(d) - BLOCK - 1,
        200
    )

    val_windows[name] = np.stack([
        d[
            i:i + BLOCK + 1
        ].astype(np.int64)
        for i in starts
    ])

print("Validation windows ready.")
print()

# ============================================================
# 9. TRAIN STEP
# ============================================================

def train_step(lr):

    for g in opt.param_groups:
        g["lr"] = lr

    total_loss = 0.0

    for _ in range(GA):

        x, y = get_batch()

        with torch.autocast(
            "cuda",
            dtype=torch.float16
        ):
            logits = model(x)

        loss = F.cross_entropy(
            logits.float().reshape(-1, V),
            y.reshape(-1)
        )

        loss = loss / GA

        scaler.scale(loss).backward()

        total_loss += loss.item()

        del x, y, logits, loss

    scaler.unscale_(opt)

    grad_norm = nn.utils.clip_grad_norm_(
        model.parameters(),
        CFG["CLIP"]
    )

    scaler.step(opt)
    scaler.update()

    opt.zero_grad(
        set_to_none=True
    )

    return (
        total_loss,
        float(grad_norm)
    )

# ============================================================
# 10. VALIDATION
# ============================================================

@torch.no_grad()
def val_loss(bs=1):

    model.eval()

    results = {}

    for name, windows in val_windows.items():

        total = 0.0
        count = 0

        for i in range(
            0,
            len(windows),
            bs
        ):

            b = torch.from_numpy(
                windows[i:i + bs]
            ).to(dev)

            with torch.autocast(
                "cuda",
                dtype=torch.float16
            ):
                logits = model(
                    b[:, :-1]
                )

            total += F.cross_entropy(
                logits.float().reshape(-1, V),
                b[:, 1:].reshape(-1),
                reduction="sum"
            ).item()

            count += b[:, 1:].numel()

            del b, logits

        results[name] = total / count

    model.train()

    return results

# ============================================================
# 11. CHECKPOINT
# ============================================================

def save_ckpt(path, step, best):

    torch.save(
        {
            "model": model.state_dict(),
            "opt": opt.state_dict(),
            "scaler": scaler.state_dict(),
            "step": step,
            "best": best,
            "cfg": CFG,
            "n_params": n_params,
        },
        path
    )

# ============================================================
# 12. REAL TRAINING
# ============================================================

def run_training():

    step0 = 0
    best = float("inf")

    latest_path = f"{RUN}/latest.pt"

    # Resume if this run already contains a checkpoint
    if os.path.exists(latest_path):

        print("=" * 70)
        print("CHECKPOINT FOUND — RESUMING")
        print("=" * 70)

        ck = torch.load(
            latest_path,
            map_location=dev
        )

        model.load_state_dict(
            ck["model"]
        )

        opt.load_state_dict(
            ck["opt"]
        )

        scaler.load_state_dict(
            ck["scaler"]
        )

        step0 = ck["step"]
        best = ck["best"]

        print("Resume step:", step0)
        print("Best validation:", best)
        print()

    else:

        print("=" * 70)
        print("STARTING FROM FRESH RANDOM WEIGHTS")
        print("=" * 70)
        print()

    model.train()

    run_start = time.time()
    last_log_time = run_start
    last_log_step = step0

    history = []

    for step in range(
        step0,
        CFG["STEPS"]
    ):

        lr = lr_at(step)

        loss, grad_norm = train_step(lr)

        history.append(loss)

        elapsed = (
            time.time()
            - run_start
        )

        timeout = (
            elapsed >= TIME_LIMIT_H * 3600
        )

        evaluate = (
            (step + 1) % CFG["EVAL_EVERY"] == 0
            or (step + 1) == CFG["STEPS"]
            or timeout
        )

        if evaluate:

            val = val_loss(bs=1)

            val_mean = float(
                np.mean(
                    list(val.values())
                )
            )

            now = time.time()

            dt = max(
                now - last_log_time,
                1e-9
            )

            delta_steps = (
                step + 1
                - last_log_step
            )

            tok_per_sec = (
                delta_steps
                * MB
                * GA
                * BLOCK
                / dt
            )

            tokens_seen = (
                (step + 1)
                * MB
                * GA
                * BLOCK
            )

            print()
            print(
                "-" * 70
            )
            print(
                f"STEP {step+1:,}/{CFG['STEPS']:,}"
            )
            print(
                f"TOKENS: {tokens_seen/1e6:,.2f}M"
            )
            print(
                f"TRAIN LOSS: {np.mean(history[-50:]):.4f}"
            )

            for k, v in val.items():
                print(
                    f"{k}: {v:.4f}"
                )

            print(
                f"VAL MEAN: {val_mean:.4f}"
            )
            print(
                f"LR: {lr:.6e}"
            )
            print(
                f"GRAD NORM: {grad_norm:.4f}"
            )
            print(
                f"THROUGHPUT: {tok_per_sec:,.0f} tok/s"
            )

            if torch.cuda.is_available():
                print(
                    f"GPU MEMORY: "
                    f"{torch.cuda.max_memory_allocated()/1e9:.2f} GB"
                )

            # Save experiment log
            record = {
                "step": step + 1,
                "tokens": int(tokens_seen),
                "train_loss": float(
                    np.mean(history[-50:])
                ),
                "val_mean": val_mean,
                "lr": float(lr),
                "grad_norm": float(grad_norm),
                "tok_per_sec": float(tok_per_sec),
            }

            record.update(val)

            with open(
                f"{RUN}/log.jsonl",
                "a"
            ) as f:
                f.write(
                    json.dumps(record)
                    + "\n"
                )

            # Best checkpoint
            if val_mean < best:

                best = val_mean

                save_ckpt(
                    f"{RUN}/best.pt",
                    step + 1,
                    best
                )

                print(
                    f"BEST CHECKPOINT SAVED"
                    f" | val={best:.4f}"
                )

            # Latest checkpoint
            save_ckpt(
                f"{RUN}/latest.pt",
                step + 1,
                best
            )

            print(
                f"LATEST CHECKPOINT SAVED"
                f" | step={step+1:,}"
            )

            last_log_time = now
            last_log_step = step + 1

        # ----------------------------------------------------
        # TIME GUARD
        # ----------------------------------------------------

        if timeout:

            print()
            print("=" * 70)
            print("8.25-HOUR TIME GUARD REACHED")
            print("=" * 70)
            print(
                "Checkpoint saved. Resume with the same RUN path."
            )

            return "PAUSED"

    print()
    print("=" * 70)
    print("TRAINING COMPLETE")
    print("=" * 70)

    return "DONE"

# ============================================================
# 13. START REAL RUN
# ============================================================

print()
print("STARTING REAL TRAINING...")
print()

status = run_training()

print()
print("=" * 70)
print("FINAL STATUS:", status)
print("=" * 70)
print("RUN DIRECTORY:", RUN)

for f in [
    "best.pt",
    "latest.pt",
    "log.jsonl"
]:
    path = os.path.join(RUN, f)
    print(
        f"{f:12s}:",
        "OK" if os.path.exists(path) else "MISSING"
    )

PHASE 1 — REAL TRAINING
GPU: Tesla T4
VRAM: 15.64 GB
BLOCK: 1024
MICRO_BATCH: 1
GRAD_ACCUM: 16
TOKENS / OPTIMIZER STEP: 16384
TOTAL STEPS: 10000
TIME LIMIT: 8.25 hours

TRAIN TOKENS
acts : 145,554,561
sc   : 249,782,556
tri  : 137,567,074

Building model...
PARAMETERS: 88,210,944
PARAMETERS: 88.21M
Validation windows ready.


STARTING REAL TRAINING...

STARTING FROM FRESH RANDOM WEIGHTS


----------------------------------------------------------------------
STEP 500/10,000
TOKENS: 8.19M
TRAIN LOSS: 11.9810
acts_val: 10.9818
sc_val: 10.8601
tribunals_val_f: 12.2018
VAL MEAN: 11.3479
LR: 3.000000e-04
GRAD NORM: 10.0529
THROUGHPUT: 12,913 tok/s
GPU MEMORY: 2.92 GB
BEST CHECKPOINT SAVED | val=11.3479
LATEST CHECKPOINT SAVED | step=500

----------------------------------------------------------------------
STEP 1,000/10,000
TOKENS: 16.38M
TRAIN LOSS: 6.7746
acts_val: 6.5442
sc_val: 6.6508
tribunals_val_f: 7.1010
VAL MEAN: 6.7653
LR: 2.981661e-04
GRAD NORM: 5.8473
THROUGHPUT: 12,948 tok/s
G